# 세션1 — Sparse / Dense / Hybrid Retriever

지금까지는 벡터 검색(Dense Retriever) 하나만 써왔습니다. 이번 시간에는 "검색기 종류에 따라 같은 질문의 결과가 완전히 달라진다"는 걸 직접 확인합니다.

1. **BM25 (Sparse Retriever)** — 단어가 얼마나 겹치는지로 점수를 매김
2. **Dense Retriever** — 문장의 의미(임베딩)가 얼마나 가까운지로 점수를 매김
3. **Hybrid Search** — 두 점수를 합쳐서 서로의 약점을 보완

각 방식마다 **희망편(잘 찾는 경우)**과 **절망편(못 찾는 경우)**을 먼저 보고, 왜 그런 차이가 나는지 확인한 뒤 Hybrid로 넘어갑니다. 저장소 루트에 있는 키오스크 이용실태 조사 PDF와 vectorstore(모든 세션 공유)를 그대로 재사용합니다.

In [1]:
import os

from langchain_community.document_loaders import PyPDFLoader
from langchain_ollama import OllamaEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

# PDF와 vectorstore 모두 여러 세션 노트북이 공유하는 저장소 루트(data/, vectorstore/)를 그대로 씁니다.
DATA_PATH = os.path.join("..", "data", "키오스크(무인정보단말기) 이용실태 조사.pdf")
VECTORSTORE_DIR = os.path.join("..", "vectorstore")
embeddings = OllamaEmbeddings(model="qwen3-embedding:0.6b")

# BM25는 원본 문서를 직접 청킹해서 씁니다. ingest.py와 같은 설정(chunk_size=1500)으로 맞춰야
# 뒤에서 Dense(이미 이 설정으로 만들어진 vectorstore)와 공정하게 비교할 수 있습니다.
docs = PyPDFLoader(DATA_PATH).load()
splitter = RecursiveCharacterTextSplitter(chunk_size=1500, chunk_overlap=200)
splited_docs = splitter.split_documents(docs)
print(f"원본 페이지 수: {len(docs)}, 청크 수: {len(splited_docs)}")


def show_results(docs, title):
    """검색 결과를 순위와 함께 짧게 출력합니다."""
    print(f"--- {title} ---")
    for i, doc in enumerate(docs):
        print(f"[{i + 1}순위] {doc.page_content[:80].strip().replace(chr(10), ' ')}...")
    print()

/var/folders/px/v7_qzrl907d919ql0sn3f74r0000gn/T/ipykernel_73725/1017673376.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


원본 페이지 수: 59, 청크 수: 73


## ① BM25 (Sparse Retriever)

BM25는 "질문과 문서에 같은 단어가 얼마나 겹치는지"로 점수를 매기는 전통적인 검색 알고리즘입니다. 의미는 전혀 모르고, 오직 **단어(토큰)가 일치하는지**만 봅니다.

`rank_bm25` 라이브러리를 감싼 LangChain의 `BM25Retriever`를 씁니다.

In [2]:
from langchain_community.retrievers import BM25Retriever

bm25_retriever = BM25Retriever.from_documents(splited_docs)
bm25_retriever.k = 3

### BM25 희망편 — 문서에만 나오는 고유 용어로 물어보면?

36페이지에는 이런 내용이 있습니다.

> \* 키오스크 배려 존(Zone) - (목적) 키오스크 이용이 익숙지 않은 소비자가 뒤의 대기 줄로 인한 심리적 압박감을 덜 느끼도록 ...

"배려 존"은 이 보고서에서만 쓰이는 고유 용어라, 일상 대화에 잘 안 나옵니다. 이런 용어를 그대로 묻는 자연스러운 질문 **"키오스크 배려존이 뭐야?"**로 검색해봅니다. 단어 자체가 문서와 겹치니 BM25가 유리할 거라 예상할 수 있습니다.

In [3]:
query_bm25_hope = "키오스크 배려존이 뭐야?"

bm25_hope_results = bm25_retriever.invoke(query_bm25_hope)
show_results(bm25_hope_results, f"BM25 희망편: '{query_bm25_hope}'")
print("→ 2순위로 36페이지(키오스크 배려 존) 청크를 찾았습니다. 'Zone'이라는 고유 용어가 그대로 겹칩니다.")

--- BM25 희망편: '키오스크 배려존이 뭐야?' ---
[1순위] 키오스크(무인정보단말기)이용 실태조사 9 시장조사국 시장감시팀 □(시장규모)세계 키오스크 시장은 ’20년 기준 176.3억 달러(한화로 약 23...
[2순위] 키오스크(무인정보단말기)이용 실태조사 36 시장조사국 시장감시팀 * 키오스크 배려 존(Zone) - (목적) 키오스크 이용이 익숙지 않은 소비자...
[3순위] 키오스크(무인정보단말기)이용 실태조사 22 시장조사국 시장감시팀 Ⅴ소비자 설문조사▣ 설문대상: 키오스크 이용 경험이 있는 소비자 500명을 연령...

→ 2순위로 36페이지(키오스크 배려 존) 청크를 찾았습니다. 'Zone'이라는 고유 용어가 그대로 겹칩니다.


### BM25 절망편 — 표 안 수치를 자연스러운 질문으로 물어보면?

10페이지 [표2-1-2]에는 이런 행이 있습니다.

> 가. 무인행정민원 3,904대 4,613대 (2019년 / 2021년)

이 표를 보지 않고, 궁금해서 물어볼 법한 자연스러운 질문을 던져보겠습니다: **"2019년 무인행정민원 운영대수가 몇 대야?"**

문서 원문은 "가. 무인행정민원3,904대4,613대"처럼 단어들이 띄어쓰기 없이 붙어 있어서, 질문의 단어들과 거의 겹치지 않습니다.

In [4]:
query_bm25_despair = "2019년 무인행정민원 운영대수가 몇 대야?"

bm25_despair_results = bm25_retriever.invoke(query_bm25_despair)
show_results(bm25_despair_results, f"BM25 절망편: '{query_bm25_despair}'")
print("→ 10페이지([표2-1-2] 국내 키오스크 보급 현황) 청크가 안 보입니다. 표 안 단어들이 띄어쓰기 없이 붙어 있어 질문과 겹치지 않으면 BM25는 못 찾습니다.")

--- BM25 절망편: '2019년 무인행정민원 운영대수가 몇 대야?' ---
[1순위] 키오스크(무인정보단말기)이용 실태조사 20 시장조사국 시장감시팀 □(불만·피해 발생 원인)소비자 불만·피해 발생 원인을분석한 결과,‘기기 오작동...
[2순위] 키오스크(무인정보단말기)이용 실태조사 21 시장조사국 시장감시팀 □이중 결제[사례1]기기 오작동-2018년 4월, 소비자는 커피전문점 무인주문기...
[3순위] 키오스크(무인정보단말기)이용 실태조사 59 시장조사국 시장감시팀 ㅇ비마이너,「장애인은 키오스크 3년 뒤에 써라?또 ‘단계적 적용’논란」,2022...

→ 10페이지([표2-1-2] 국내 키오스크 보급 현황) 청크가 안 보입니다. 표 안 단어들이 띄어쓰기 없이 붙어 있어 질문과 겹치지 않으면 BM25는 못 찾습니다.


## ② Dense Retriever (임베딩 기반)

Dense Retriever는 단어가 겹치는지가 아니라, 문장을 벡터로 바꿔서 **의미가 얼마나 가까운지**로 검색합니다. 저장소 루트의 공유 vectorstore(`../vectorstore`)를 그대로 불러와 씁니다.

In [5]:
from langchain_chroma import Chroma

vectorstore = Chroma(persist_directory=VECTORSTORE_DIR, embedding_function=embeddings)
dense_retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

### Dense 희망편 — BM25가 놓쳤던 자연스러운 질문

방금 BM25가 놓쳤던 질문 `"2019년 무인행정민원 운영대수가 몇 대야?"`를 그대로 Dense Retriever에 넣어봅니다. 단어가 정확히 겹치지 않아도, "2019년 무인행정민원 키오스크 대수를 묻는 질문"이라는 **의미**는 표 내용과 같으니 잘 찾을 거라 예상할 수 있습니다.

In [6]:
dense_hope_results = dense_retriever.invoke(query_bm25_despair)
show_results(dense_hope_results, f"Dense 희망편: '{query_bm25_despair}'")
print("→ 1순위로 10페이지([표2-1-2] 국내 키오스크 보급 현황) 청크를 정확히 찾았습니다. BM25가 놓친 걸 Dense가 찾아냅니다.")

--- Dense 희망편: '2019년 무인행정민원 운영대수가 몇 대야?' ---
[1순위] 키오스크(무인정보단말기)이용 실태조사 10 시장조사국 시장감시팀 [표2-1-2] 국내 키오스크 보급 현황(추정)분야운영대수(추정)2019년202...
[2순위] 이상(19명)10.5%5.3%26.3%52.6%5.3%3.37점병원(194명)20대(25명)0.0%8.0%24.0%52.0%16.0%3.76점3...
[3순위] 키오스크(무인정보단말기)이용 실태조사 20 시장조사국 시장감시팀 □(불만·피해 발생 원인)소비자 불만·피해 발생 원인을분석한 결과,‘기기 오작동...

→ 1순위로 10페이지([표2-1-2] 국내 키오스크 보급 현황) 청크를 정확히 찾았습니다. BM25가 놓친 걸 Dense가 찾아냅니다.


### Dense 절망편 — BM25가 잘 찾았던 고유 용어 검색

이번엔 반대로, BM25가 찾았던 질문 `"키오스크 배려존이 뭐야?"`을 Dense Retriever에 넣어봅니다.

"배려 존"은 이 문서 밖에서는 거의 안 쓰이는 용어라, 임베딩 모델이 이 단어의 의미를 "키오스크 편의 시설"과 정확히 연결 짓기 어려울 수 있습니다. Dense가 이걸 잘 찾을 수 있을까요?

In [7]:
dense_despair_results = dense_retriever.invoke(query_bm25_hope)
show_results(dense_despair_results, f"Dense 절망편: '{query_bm25_hope}'")
print("→ 36페이지(키오스크 배려 존) 청크가 3위 안에 없습니다. 이 문서에서만 쓰이는 고유 용어는 임베딩이 잘 못 잡아냅니다.")

--- Dense 절망편: '키오스크 배려존이 뭐야?' ---
[1순위] 키오스크(무인정보단말기)이용 실태조사 35 시장조사국 시장감시팀 ㅇ (③큰 화면‧ 글씨,음성 안내 등 적용)큰 화면 및 글씨,음성 안내 추가 등...
[2순위] 키오스크(무인정보단말기)이용 실태조사 7 시장조사국 시장감시팀 Ⅱ일반 현황1. 키오스크(무인정보단말기)의 정의 및 시장동향□(정의)이용자의 조작...
[3순위] · · · · · · · · · · · · · · · · · 35[표5-3-11] ‘키오스크 기능 표준화’ 필요성(N=500) · · · · ·...

→ 36페이지(키오스크 배려 존) 청크가 3위 안에 없습니다. 이 문서에서만 쓰이는 고유 용어는 임베딩이 잘 못 잡아냅니다.


## ③ 정리 및 Hybrid Search

방금 확인한 내용을 정리하면:

| 쿼리 | BM25 | Dense |
|---|---|---|
| `"키오스크 배려존이 뭐야?"` (고유 용어) | ✅ 2순위 정답 | ❌ 3위 안에 없음 |
| `"2019년 무인행정민원 운영대수가 몇 대야?"` (자연스러운 질문) | ❌ 3위 안에 없음 | ✅ 1순위 정답 |

**두 방식은 서로 다른 이유로 실패합니다.** BM25는 단어가 안 겹치면 못 찾고, Dense는 문서에서만 쓰이는 낯선 용어의 의미를 잘 못 잡아냅니다. 그래서 실무에서는 둘 중 하나만 쓰지 않고 **점수를 합쳐서(Hybrid)** 씁니다.

`EnsembleRetriever`는 여러 retriever의 결과를 RRF(Reciprocal Rank Fusion) 방식으로 합쳐서 순위를 다시 매깁니다. 각 retriever가 매긴 순위(1위, 2위, ...)를 점수로 환산해 합산하는 방식이라, 점수 스케일이 다른 BM25와 Dense도 공정하게 합칠 수 있습니다. `weights`로 각 retriever의 비중을 조절할 수 있습니다.

In [8]:
from langchain_classic.retrievers import EnsembleRetriever

hybrid_retriever = EnsembleRetriever(
    retrievers=[bm25_retriever, dense_retriever],
    weights=[0.5, 0.5],
)

sample = hybrid_retriever.invoke("키오스크 시장규모")
show_results(sample, "Hybrid 동작 확인")

--- Hybrid 동작 확인 ---
[1순위] 키오스크(무인정보단말기)이용 실태조사 9 시장조사국 시장감시팀 □(시장규모)세계 키오스크 시장은 ’20년 기준 176.3억 달러(한화로 약 23...
[2순위] 키오스크(무인정보단말기)이용 실태조사 22 시장조사국 시장감시팀 Ⅴ소비자 설문조사▣ 설문대상: 키오스크 이용 경험이 있는 소비자 500명을 연령...
[3순위] 키오스크(무인정보단말기)이용 실태조사 36 시장조사국 시장감시팀 * 키오스크 배려 존(Zone) - (목적) 키오스크 이용이 익숙지 않은 소비자...
[4순위] 키오스크(무인정보단말기)이용 실태조사 52 시장조사국 시장감시팀 ㅇ 그러나 「장차법」시행일인’23년 1월 28일 이전에 설치한 키오스크의 경우...



### 절망편 두 개를 Hybrid로 다시 검색

각 retriever가 단독으로는 놓쳤던 두 쿼리를 Hybrid로 다시 실행해봅니다. 둘 중 한쪽이라도 상위권에서 찾아낸 청크라면, Hybrid 결과에도 상위권으로 살아남아야 합니다.

In [9]:
# EnsembleRetriever는 각 retriever의 k(여기선 3)를 합친 뒤 순위를 다시 매기므로,
# 결과 개수가 3보다 많아질 수 있습니다. 상위 몇 개만 확인합니다.
hybrid_hope = hybrid_retriever.invoke(query_bm25_hope)
show_results(hybrid_hope[:4], f"Hybrid: '{query_bm25_hope}' (BM25는 성공, Dense는 실패했던 쿼리)")

hybrid_despair = hybrid_retriever.invoke(query_bm25_despair)
show_results(hybrid_despair[:4], f"Hybrid: '{query_bm25_despair}' (Dense는 성공, BM25는 실패했던 쿼리)")

print("→ 두 쿼리 모두 정답 청크가 상위권에 남아 있습니다. 한쪽이 놓쳐도 다른 쪽이 보완해줍니다.")

--- Hybrid: '키오스크 배려존이 뭐야?' (BM25는 성공, Dense는 실패했던 쿼리) ---
[1순위] 키오스크(무인정보단말기)이용 실태조사 9 시장조사국 시장감시팀 □(시장규모)세계 키오스크 시장은 ’20년 기준 176.3억 달러(한화로 약 23...
[2순위] 키오스크(무인정보단말기)이용 실태조사 35 시장조사국 시장감시팀 ㅇ (③큰 화면‧ 글씨,음성 안내 등 적용)큰 화면 및 글씨,음성 안내 추가 등...
[3순위] 키오스크(무인정보단말기)이용 실태조사 36 시장조사국 시장감시팀 * 키오스크 배려 존(Zone) - (목적) 키오스크 이용이 익숙지 않은 소비자...
[4순위] 키오스크(무인정보단말기)이용 실태조사 7 시장조사국 시장감시팀 Ⅱ일반 현황1. 키오스크(무인정보단말기)의 정의 및 시장동향□(정의)이용자의 조작...



--- Hybrid: '2019년 무인행정민원 운영대수가 몇 대야?' (Dense는 성공, BM25는 실패했던 쿼리) ---
[1순위] 키오스크(무인정보단말기)이용 실태조사 20 시장조사국 시장감시팀 □(불만·피해 발생 원인)소비자 불만·피해 발생 원인을분석한 결과,‘기기 오작동...
[2순위] 키오스크(무인정보단말기)이용 실태조사 10 시장조사국 시장감시팀 [표2-1-2] 국내 키오스크 보급 현황(추정)분야운영대수(추정)2019년202...
[3순위] 키오스크(무인정보단말기)이용 실태조사 21 시장조사국 시장감시팀 □이중 결제[사례1]기기 오작동-2018년 4월, 소비자는 커피전문점 무인주문기...
[4순위] 이상(19명)10.5%5.3%26.3%52.6%5.3%3.37점병원(194명)20대(25명)0.0%8.0%24.0%52.0%16.0%3.76점3...

→ 두 쿼리 모두 정답 청크가 상위권에 남아 있습니다. 한쪽이 놓쳐도 다른 쪽이 보완해줍니다.


## 정리

- **BM25(Sparse)**: 단어가 겹치는지만 봅니다. 문서에만 나오는 고유 용어를 그대로 묻는 질문처럼 "단어가 정확히 일치하는" 경우에 강하지만, 표현이 조금만 달라지거나 띄어쓰기가 다르면 못 찾습니다.
- **Dense**: 문장의 의미로 검색합니다. 자연스러운 질문(패러프레이즈)에 강하지만, 이 문서에서만 쓰이는 낯선 용어처럼 의미를 학습하지 못한 표현은 잘 못 찾습니다.
- **Hybrid(EnsembleRetriever)**: 두 retriever의 순위를 합쳐서, 한쪽이 놓친 걸 다른 쪽이 보완하게 만듭니다.